# Phase 3 — Data Preprocessing & Feature Engineering

Steps:
1. Tokenize titles, build vocabulary
2. Load GloVe → embedding matrix
3. Encode all news titles as integer sequences
4. Parse behaviors into (history, candidates, labels) samples
5. Save processed data as `.pkl` for fast reloading

In [4]:
import pandas as pd
import numpy as np
import pickle
import os
import random
import sys
sys.path.insert(0, '../src')
%cd C:\Users\lasc2\OneDrive\Desktop\News_Recommend
from collections import Counter

import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Paths
TRAIN_DIR  = 'data/MINDsmall_train'
DEV_DIR    = 'data/MINDsmall_dev'
GLOVE_PATH = 'data/glove/glove.6B.300d.txt'
OUT_DIR    = 'data/processed'
os.makedirs(OUT_DIR, exist_ok=True)

# Hyperparameters
MAX_TITLE_LEN = 30
MIN_WORD_FREQ = 2
EMBED_DIM     = 300
MAX_HISTORY   = 50
NEG_K         = 4

print('Config loaded.')

C:\Users\lasc2\OneDrive\Desktop\News_Recommend
Config loaded.


## 1. Load Raw Data

In [5]:
NEWS_COLS = ['news_id', 'category', 'subcategory', 'title',
             'abstract', 'url', 'title_entities', 'abstract_entities']
BEH_COLS  = ['impression_id', 'user_id', 'time', 'history', 'impressions']

news_df   = pd.read_csv(f'{TRAIN_DIR}/news.tsv',      sep='\t', names=NEWS_COLS)
beh_train = pd.read_csv(f'{TRAIN_DIR}/behaviors.tsv', sep='\t', names=BEH_COLS)
beh_dev   = pd.read_csv(f'{DEV_DIR}/behaviors.tsv',   sep='\t', names=BEH_COLS)

# Also load dev news (may include articles not in train)
news_dev  = pd.read_csv(f'{DEV_DIR}/news.tsv', sep='\t', names=NEWS_COLS)
news_all  = pd.concat([news_df, news_dev]).drop_duplicates('news_id').reset_index(drop=True)

print(f'Combined news articles: {len(news_all):,}')

Combined news articles: 65,238


## 2. Tokenizer & Vocabulary

In [6]:
class NewsTokenizer:
    def __init__(self, max_title_len=30, min_word_freq=2):
        self.max_title_len = max_title_len
        self.min_word_freq = min_word_freq
        self.word2idx = {'<PAD>': 0, '<UNK>': 1}

    def _tokenize(self, text):
        if pd.isna(text) or not isinstance(text, str):
            return []
        return nltk.word_tokenize(text.lower())

    def build_vocab(self, titles):
        word_counts = Counter()
        for title in titles:
            word_counts.update(self._tokenize(title))
        for word, count in word_counts.items():
            if count >= self.min_word_freq:
                self.word2idx[word] = len(self.word2idx)
        print(f'Vocabulary size: {len(self.word2idx):,}')

    def encode_title(self, title):
        tokens  = self._tokenize(title)
        indices = [self.word2idx.get(t, 1) for t in tokens]
        # Truncate or pad
        indices = indices[:self.max_title_len]
        indices += [0] * (self.max_title_len - len(indices))
        return indices


tokenizer = NewsTokenizer(MAX_TITLE_LEN, MIN_WORD_FREQ)
# Build vocab only on train titles to prevent leakage
tokenizer.build_vocab(news_df['title'].tolist())

Vocabulary size: 20,774


## 3. Load GloVe → Embedding Matrix

In [7]:
def load_glove(glove_path, word2idx, embed_dim=300):
    vocab_size = len(word2idx)
    # Small random init; PAD stays zero
    matrix = (np.random.randn(vocab_size, embed_dim) * 0.01).astype('float32')
    matrix[0] = 0.0

    found = 0
    with open(glove_path, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.rstrip().split(' ')
            word  = parts[0]
            if word in word2idx:
                matrix[word2idx[word]] = np.array(parts[1:], dtype='float32')
                found += 1

    print(f'GloVe coverage: {found}/{vocab_size} words ({found/vocab_size*100:.1f}%)')
    return matrix


embedding_matrix = load_glove(GLOVE_PATH, tokenizer.word2idx, EMBED_DIM)
print(f'Embedding matrix shape: {embedding_matrix.shape}')

GloVe coverage: 19069/20774 words (91.8%)
Embedding matrix shape: (20774, 300)


## 4. Encode All News Titles

In [8]:
# news_encoded: dict { news_id -> list[int] of length MAX_TITLE_LEN }
news_encoded = {}
for _, row in news_all.iterrows():
    news_encoded[row['news_id']] = tokenizer.encode_title(row['title'])

# Also store category mapping
categories   = sorted(news_all['category'].dropna().unique())
cat2idx      = {c: i+1 for i, c in enumerate(categories)}  # 0 = unknown
news_cat     = {row['news_id']: cat2idx.get(row['category'], 0)
                for _, row in news_all.iterrows()}

# Placeholder encoding for unknown article IDs
PAD_TITLE    = [0] * MAX_TITLE_LEN

print(f'Encoded {len(news_encoded):,} articles')
print(f'Categories: {categories}')

Encoded 65,238 articles
Categories: ['autos', 'entertainment', 'finance', 'foodanddrink', 'games', 'health', 'kids', 'lifestyle', 'middleeast', 'movies', 'music', 'news', 'northamerica', 'sports', 'travel', 'tv', 'video', 'weather']


## 5. Parse Behaviors → Training Samples

In [9]:
def parse_behaviors_train(behaviors_df, news_encoded, pad_title,
                           max_history=50, neg_k=4, seed=42):
    """
    Returns list of dicts:
      history    : np.array (max_history, MAX_TITLE_LEN)  — padded click history
      hist_mask  : np.array (max_history,)                — 1 where real article
      candidates : np.array (1+neg_k, MAX_TITLE_LEN)     — [pos, neg1, ..., negK]
      labels     : [1, 0, 0, ...]                         — always pos at index 0
    """
    rng     = np.random.default_rng(seed)
    samples = []

    for _, row in behaviors_df.iterrows():
        # Parse history
        hist_ids  = row['history'].split() if pd.notna(row['history']) and str(row['history']).strip() else []
        hist_ids  = hist_ids[-max_history:]   # keep most recent
        hist_enc  = [news_encoded.get(nid, pad_title) for nid in hist_ids]

        # Pad history
        hist_len  = len(hist_enc)
        pad_count = max_history - hist_len
        hist_enc  = hist_enc + [pad_title] * pad_count
        hist_mask = [1] * hist_len + [0] * pad_count

        # Parse impressions
        if pd.isna(row['impressions']) or not str(row['impressions']).strip():
            continue

        pos_ids, neg_ids = [], []
        for item in str(row['impressions']).split():
            nid, label = item.rsplit('-', 1)
            if label == '1':
                pos_ids.append(nid)
            else:
                neg_ids.append(nid)

        if not pos_ids or not neg_ids:
            continue

        # One sample per positive click
        for pos_id in pos_ids:
            k_actual  = min(neg_k, len(neg_ids))
            sampled   = rng.choice(neg_ids, size=k_actual, replace=False).tolist()
            cand_ids  = [pos_id] + sampled
            # Pad candidates if fewer than neg_k negatives available
            while len(cand_ids) < 1 + neg_k:
                cand_ids.append(pos_id)   # duplicate pos as dummy (masked in loss)

            candidates = [news_encoded.get(cid, pad_title) for cid in cand_ids]

            samples.append({
                'history'   : np.array(hist_enc,   dtype=np.int32),
                'hist_mask' : np.array(hist_mask,  dtype=np.int32),
                'candidates': np.array(candidates, dtype=np.int32),
                'labels'    : [1] + [0] * (len(cand_ids) - 1),
            })

    return samples


print('Parsing train behaviors...')
train_samples = parse_behaviors_train(beh_train, news_encoded, PAD_TITLE,
                                       MAX_HISTORY, NEG_K, SEED)
print(f'Train samples: {len(train_samples):,}')

Parsing train behaviors...
Train samples: 236,344


In [10]:
def parse_behaviors_dev(behaviors_df, news_encoded, pad_title, max_history=50):
    """
    For evaluation: keep ALL candidates per impression (no negative sampling).
    Returns list of dicts with full impression candidate list and labels.
    """
    samples = []
    for _, row in behaviors_df.iterrows():
        hist_ids  = row['history'].split() if pd.notna(row['history']) and str(row['history']).strip() else []
        hist_ids  = hist_ids[-max_history:]
        hist_enc  = [news_encoded.get(nid, pad_title) for nid in hist_ids]
        hist_len  = len(hist_enc)
        pad_count = max_history - hist_len
        hist_enc  = hist_enc + [pad_title] * pad_count
        hist_mask = [1] * hist_len + [0] * pad_count

        if pd.isna(row['impressions']) or not str(row['impressions']).strip():
            continue

        cand_ids, labels = [], []
        for item in str(row['impressions']).split():
            nid, label = item.rsplit('-', 1)
            cand_ids.append(nid)
            labels.append(int(label))

        if sum(labels) == 0 or sum(labels) == len(labels):
            continue   # skip degenerate impressions

        candidates = [news_encoded.get(cid, pad_title) for cid in cand_ids]

        samples.append({
            'history'   : np.array(hist_enc,   dtype=np.int32),
            'hist_mask' : np.array(hist_mask,  dtype=np.int32),
            'candidates': candidates,    # variable length — kept as list
            'labels'    : labels,
        })

    return samples


print('Parsing dev behaviors...')
dev_samples = parse_behaviors_dev(beh_dev, news_encoded, PAD_TITLE, MAX_HISTORY)
print(f'Dev samples: {len(dev_samples):,}')

Parsing dev behaviors...
Dev samples: 73,152


## 6. Save Processed Data

In [11]:
with open(f'{OUT_DIR}/train_samples.pkl', 'wb') as f:
    pickle.dump(train_samples, f)

with open(f'{OUT_DIR}/dev_samples.pkl', 'wb') as f:
    pickle.dump(dev_samples, f)

np.save(f'{OUT_DIR}/embedding_matrix.npy', embedding_matrix)

with open(f'{OUT_DIR}/tokenizer.pkl', 'wb') as f:
    pickle.dump(tokenizer, f)

with open(f'{OUT_DIR}/news_encoded.pkl', 'wb') as f:
    pickle.dump(news_encoded, f)

print('All processed data saved to data/processed/')
print(f'  embedding_matrix : {embedding_matrix.shape}')
print(f'  train_samples    : {len(train_samples):,}')
print(f'  dev_samples      : {len(dev_samples):,}')

All processed data saved to data/processed/
  embedding_matrix : (20774, 300)
  train_samples    : 236,344
  dev_samples      : 73,152


## 7. Sanity Check

In [12]:
sample = train_samples[0]
print('Sample keys         :', list(sample.keys()))
print('history shape       :', sample['history'].shape)
print('hist_mask shape     :', sample['hist_mask'].shape)
print('candidates shape    :', sample['candidates'].shape)
print('labels              :', sample['labels'])
print('Max candidate index :', sample['candidates'].max())
print('Vocab size          :', len(tokenizer.word2idx))

Sample keys         : ['history', 'hist_mask', 'candidates', 'labels']
history shape       : (50, 30)
hist_mask shape     : (50,)
candidates shape    : (5, 30)
labels              : [1, 0, 0, 0, 0]
Max candidate index : 15678
Vocab size          : 20774
